In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# CrackSegDiff: Simplified Training & Inference
Automated setup, data preparation (First 500 Test / 2000 Train), training, and testing.

In [ ]:
# 1. Setup Environment & Weights
# @title Configuration
TRAIN_MODEL = False # @param {type:"boolean"}
DRIVE_MODEL_DIR = "/content/drive/MyDrive/CrackSegDiff/models"

import os
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

!nvidia-smi
import os
if not os.path.exists('CrackSegDiff'):
    !git clone https://github.com/Ludwig-H/CrackSegDiff.git
%cd CrackSegDiff
!git pull

target_file_gd = 'CrackSegDiff/guided_diffusion/gaussian_diffusion.py'
if os.path.exists(target_file_gd):
    with open(target_file_gd, 'r') as f: content = f.read()
    content = content.replace('beta_end = scale * 0.02', 'beta_end = scale * 0.02\n        if beta_end > 0.999: beta_end = 0.999')
    with open(target_file_gd, 'w') as f: f.write(content)

# Downgrade PyTorch to stable 2.4.0 for guaranteed Mamba compatibility
print("Installing PyTorch 2.4.0 compatible with Mamba wheels...")
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

!pip install -r requirement.txt

# Force install pre-built wheels for Mamba-SSM and Causal-Conv1d to avoid compilation errors and ensure GPU speed
print("Installing Optimized Mamba Kernels...")
import torch
cuda_version = torch.version.cuda.replace('.', '')
torch_version = torch.__version__.split('+')[0].replace('.', '')
# Assuming standard Colab PyTorch 2.x and CUDA 11.8 or 12.x
# We use the releases from Dao-AILab which are reliable
!pip install ninja  # Speeds up compilation
!pip install causal-conv1d>=1.0.0 --no-build-isolation -v
!pip install mamba-ssm>=1.0.1 --no-build-isolation -v

# If the above standard install fails (it tries to build), we could try finding wheels:
# (But usually --no-build-isolation helps or just standard pip works if env is clean)

# Verify installation
try:
    import mamba_ssm
    print("SUCCESS: Mamba SSM installed successfully! GPU acceleration enabled (x10 speed).")
except ImportError:
    print("WARNING: Mamba SSM installation failed.")
    print("Fallback to Pure Python active (SLOWER).")

# Download Pretrained Weights
!mkdir -p pretrained_weights
!gdown 1JYqMxM5dbCLZ-WGPKtIofYJhj0VPuy3l -O pretrained_weights/vssm_base_0229_ckpt_epoch_237.pth

# Patch Hardcoded Paths
target_file = 'CrackSegDiff/guided_diffusion/unet.py'
new_path = os.path.abspath('pretrained_weights/vssm_base_0229_ckpt_epoch_237.pth')
if os.path.exists(target_file):
    with open(target_file, 'r') as f: content = f.read()
    content = content.replace('/home/dell/jlc/segdiff/pre_trained_weights/vssm_base_0229_ckpt_epoch_237.pth', new_path)
    with open(target_file, 'w') as f: f.write(content)
    print("Path patched successfully.")

Tue Jan 20 12:31:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             11W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Installing Optimized Mamba Kernels...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 15.0 MB/s eta 0:00:00
  Running command Preparing metadata (pyproject.toml)


  torch.__version__  = 2.4.0+cu121


  running dist_info
  creating /tmp/pip-modern-metadata-8g166rlk/causal_conv1d.egg-info
  writing /tmp/pip-modern-metadata-8g166rlk/causal_conv1d.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-modern-metadata-8g166rlk/causal_conv1d.egg-info/dependency_links.txt
  writing requirements to /tmp/pip-modern-metadata-8g166rlk/causal_conv1d.egg-info/requires.txt
  writing top-level names to /tmp/pip-modern-metadata-8g166rlk/causal_conv1d.egg-info/top_level.txt
  writing manifest file '/tmp/pip-modern-metadata-8g166rlk/causal_conv1d.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-modern-metadata-8g166rlk/causal_conv1d.egg-info/SOURCES.txt'
  reading manifest template 'MANIFEST.in'

  adding license file 'LICENSE'
  adding license file 'AUTHORS'
  writing manifest file

In [ ]:
# 2. Prepare Data (First 500 Test / 2000 Train)
!rm -rf data && mkdir -p data
%cd data
!gdown 1qnLMCeon7LJjT9H0ENiNF5sFs-F7-NvK -O data.zip
!unzip -q -o data.zip
%cd ..

import os, glob, shutil
print("Organizing Data...")

# Find folders
try:
    src_img = glob.glob('data/**/img/fused', recursive=True)[0]
    src_lbl = glob.glob('data/**/lbs', recursive=True)[0]
except IndexError:
    # Fallback if 'fused' not found, try 'intensity'
    print("Fused folder not found, checking intensity...")
    src_img = glob.glob('data/**/img/intensity', recursive=True)[0]
    src_lbl = glob.glob('data/**/lbs', recursive=True)[0]

print(f"Images source: {src_img}")
print(f"Labels source: {src_lbl}")

# Get sorted file lists
img_files = sorted(glob.glob(os.path.join(src_img, '*')))
lbl_files = sorted(glob.glob(os.path.join(src_lbl, '*.bmp')))

# Split: First 500 Test, Rest Train
test_pairs = list(zip(img_files[:500], lbl_files[:500]))
train_pairs = list(zip(img_files[500:], lbl_files[500:]))

print(f"Test Set: {len(test_pairs)} (First 500)")
print(f"Train Set: {len(train_pairs)} (Rest)")

# Copy to formatted directories
train_dir = os.path.abspath('data/Train')
test_dir = os.path.abspath('data/Test')

for pairs, dest in [(train_pairs, train_dir), (test_pairs, test_dir)]:
    os.makedirs(os.path.join(dest, '5d'), exist_ok=True)
    os.makedirs(os.path.join(dest, 'mask'), exist_ok=True)
    for img, mask in pairs:
        shutil.copy(img, os.path.join(dest, '5d'))
        shutil.copy(mask, os.path.join(dest, 'mask'))
print("Data preparation complete.")

In [ ]:
# 3. Train Model
import os
import shutil
import glob

data_dir = os.path.abspath('data/Train')
out_dir = os.path.abspath('results/train_output')
os.makedirs(out_dir, exist_ok=True)

if TRAIN_MODEL:
    print("Starting Training...")
    !python CrackSegDiff/segmentation_train.py --data_dir {data_dir} --out_dir {out_dir} --image_size 256 --num_channels 96 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 500 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --lr 5e-5 --batch_size 8 --save_interval 5000 --lr_anneal_steps 40000

    # Save to Drive
    print(f"Backing up model to {DRIVE_MODEL_DIR}...")
    for model_file in glob.glob(os.path.join(out_dir, '*.pt')):
        shutil.copy(model_file, DRIVE_MODEL_DIR)
    print("Backup complete.")
else:
    print("Training skipped (TRAIN_MODEL=False).")

In [ ]:
# # 4. Inference
# import glob, os

# # Select Model
# if TRAIN_MODEL:
#     models = sorted(glob.glob('results/train_output/*.pt'))
#     model_path = models[-1] if models else "pretrained_weights/savedmodel100000.pt"
# else:
#     print(f"Using model from Drive: {DRIVE_MODEL_DIR}")
#     drive_models = sorted(glob.glob(os.path.join(DRIVE_MODEL_DIR, '*.pt')))
#     model_path = drive_models[-1] if drive_models else "pretrained_weights/savedmodel100000.pt"

# test_dir = os.path.abspath('data/Test')
# print(f"Using model: {model_path}")

# for modality in ['intensity', 'range', 'fused']: # , 'filtered', 'all'
#     out_path = f"results/test_output_{modality}"
#     os.makedirs(out_path, exist_ok=True)
#     print(f"Testing {modality}...")
#     !python CrackSegDiff/segmentation_sample.py --data_dir {test_dir} --out_dir {out_path} --model_path {model_path} --modality {modality} --image_size 256 --num_channels 96 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 500 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --num_ensemble 1
#     print(f"Done {modality}")

In [ ]:
# 5. Zip Results
!zip -r results.zip results

In [ ]:
# 6. Robustness Benchmark (Inference on Pre-generated Noisy Data)
# Uses noisy data from /content/drive/MyDrive/Datasets/FIND/Noisy
# Saves results to /content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff_noise

import numpy as np
import os
import glob
import shutil
from skimage import io
from tqdm import tqdm

# --- Configuration ---
NOISY_INPUT_ROOT = "/content/drive/MyDrive/Datasets/FIND/Noisy"
RESULTS_ROOT = "/content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff_noise"
TEMP_DATA_ROOT = os.path.abspath("temp_inference_noisy")
CLEAN_TEST_DIR = os.path.abspath('data/Test') # Source for masks

# Select Model (Same logic as above)
if 'TRAIN_MODEL' in locals() and not TRAIN_MODEL:
    drive_models = sorted(glob.glob(os.path.join(DRIVE_MODEL_DIR, '*.pt')))
    MODEL_PATH = drive_models[-1] if drive_models else "pretrained_weights/savedmodel100000.pt"
else:
    models = sorted(glob.glob('results/train_output/*.pt'))
    MODEL_PATH = models[-1] if models else "pretrained_weights/savedmodel100000.pt"

print(f"Using Model for Benchmark: {MODEL_PATH}")

speckle_vars = [0.3, 0.5] # [0.0, 0.01, 0.05, 0.10, 0.3, 0.5]
range_sigmas = [0.3, 0.5] # [0.0, 0.01, 0.05, 0.10, 0.3, 0.5]

experiments = [
    ("speckle_intensity", speckle_vars),
    ("gauss_range", range_sigmas),
    ("both", range_sigmas)
]
def _noise_tag(x: float, ndigits: int = 4):
    return f"{x:.{ndigits}f}".replace(".", "p")

# Ensure we have masks ready
clean_masks = sorted(glob.glob(os.path.join(CLEAN_TEST_DIR, 'mask', '*.bmp')))
clean_masks = clean_masks[:500]
if not clean_masks:
    print("WARNING: No masks found in clean test dir. Masks will be missing in temp dir.")

for exp_name, levels in experiments:
    print(f"\n=== Experiment: {exp_name} ===")

    for lvl in levels:
        lvl = float(lvl)
        tag = _noise_tag(lvl)

        # Input Directory (Drive)
        noisy_src_dir = os.path.join(NOISY_INPUT_ROOT, exp_name, tag)
        # Output Directory (Drive)
        final_dest_dir = os.path.join(RESULTS_ROOT, exp_name, tag, "test_output_fused")

        if not os.path.exists(noisy_src_dir):
            print(f"Skipping {exp_name}/{tag}: Input directory not found ({noisy_src_dir})")
            continue

        if os.path.exists(final_dest_dir) and len(glob.glob(os.path.join(final_dest_dir, '*.png'))) >= 500:
            print(f"Skipping {exp_name}/{tag}: Results already exist ({len(os.listdir(final_dest_dir)) } files)")
            continue

        print(f"Processing {exp_name} - Level {lvl} (Tag: {tag})")

        # 1. Prepare Temp Directory
        if os.path.exists(TEMP_DATA_ROOT): shutil.rmtree(TEMP_DATA_ROOT)
        temp_5d = os.path.join(TEMP_DATA_ROOT, '5d')
        temp_mask = os.path.join(TEMP_DATA_ROOT, 'mask')
        os.makedirs(temp_5d, exist_ok=True)
        os.makedirs(temp_mask, exist_ok=True)

        # 2. Reconstruct Fused Images from Drive (Intensity + Range)
        # Files are named imXXXXX_intensity.png / imXXXXX_range.png
        int_files = sorted(glob.glob(os.path.join(noisy_src_dir, '*_intensity.png')))
        print(f"Found {len(int_files)} intensity images in {noisy_src_dir}")

        count = 0
        for int_path in int_files:
            fname = os.path.basename(int_path)
            # fname format: imXXXXX_intensity.png
            base_name = fname.replace('_intensity.png', '') # imXXXXX
            range_path = os.path.join(noisy_src_dir, f"{base_name}_range.png")

            if not os.path.exists(range_path):
                # Fallback? Should not happen if generated correctly.
                continue

            # Read Images
            img_i = io.imread(int_path)
            img_r = io.imread(range_path)

            # Ensure dimensions match
            if img_i.shape[:2] != img_r.shape[:2]:
                continue

            # Stack (H, W, 3) -> R=Int, G=Range, B=0
            H, W = img_i.shape[:2]
            fused = np.zeros((H, W, 3), dtype=np.uint8)
            fused[..., 0] = img_i if img_i.ndim==2 else img_i[...,0]
            fused[..., 1] = img_r if img_r.ndim==2 else img_r[...,0]
            # Channel 2 is 0

            # Save as imXXXXX.png (Standard format for CrackSegDiff)
            # Note: The output will be imXXXXX_output_ens.png
            io.imsave(os.path.join(temp_5d, f"{base_name}.png" ), fused, check_contrast=False)
            count += 1

        # Copy Masks (Assumes matching IDs in Clean Test Dir)
        # We blindly copy all first 500 clean masks, assuming IDs match 1-500.
        for mask_path in clean_masks:
            shutil.copy(mask_path, temp_mask)

        print(f"Prepared {count} images for inference.")

        # 3. Run Inference
        temp_out_dir = os.path.join(TEMP_DATA_ROOT, "output")
        os.makedirs(temp_out_dir, exist_ok=True)

        !python CrackSegDiff/segmentation_sample.py --data_dir {TEMP_DATA_ROOT} --out_dir {temp_out_dir} --model_path {MODEL_PATH} --modality fused --image_size 256 --num_channels 96 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 500 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --num_ensemble 1

        # 4. Save to Drive
        print(f"Saving results to: {final_dest_dir}")
        os.makedirs(final_dest_dir, exist_ok=True)

        copied = 0
        for png in glob.glob(os.path.join(temp_out_dir, "*.png")):
            shutil.copy(png, final_dest_dir)
            copied += 1
        print(f"Saved {copied} result files.")

print("Robustness Benchmark (Inference Only) Complete.")